In [ ]:
!pip install mediapipe


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-contrib-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 10.2 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: opencv-contrib-python
    Found existing installation: opencv-contrib-python 4.12.0.88
    Uninstalling openc

"""
MEDIAPIPE POSE - PHÂN TÍCH CỘT SỐNG VÀ XƯƠNG BẢ VAI

Ý TƯỞNG:
---------
MediaPipe Pose cung cấp 33 landmarks trên cơ thể người, nhưng không trực tiếp
phát hiện các đốt cột sống. Chúng ta sử dụng phương pháp GEO-ANATOMICAL INFERENCE
(suy luận dựa trên giải phẫu học) để ước lượng vị trí các đốt sống và xương bả vai.

CÁC LANDMARKS MEDIAPIPE SỬ DỤNG:
---------------------------------
- Landmark 11: Vai trái (Left Shoulder)
- Landmark 12: Vai phải (Right Shoulder)  
- Landmark 23: Hông trái (Left Hip)
- Landmark 24: Hông phải (Right Hip)

PHƯƠNG PHÁP ƯỚC LƯỢNG:
----------------------
1. CỘT SỐNG:
   - C7 (Cervical 7): Điểm giữa hai vai - đốt sống cổ thấp nhất
   - T1-T6 (Thoracic upper): 1/6 đường từ vai xuống hông
   - T6-T9 (Thoracic middle): 1/3 đường từ vai xuống hông
   - T10-T12 (Thoracic lower): 2/3 đường từ vai xuống hông
   - L1-L5 (Lumbar): Vùng thắt lưng
   - S1 (Sacrum): Điểm giữa hai hông

2. XƯƠNG BẢ VAI (SCAPULA):
   Các điểm quan trọng:
   - Acromion (đỉnh vai): Tại landmark vai
   - Spine của scapula: Chạy ngang từ trong ra ngoài
   - Inferior angle (góc dưới): ~40% đường từ vai xuống hông, lệch ra ngoài
   - Medial border (mép trong): Song song với cột sống

THÔNG SỐ ĐÁNH GIÁ:
------------------
1. Confidence Score: Độ tin cậy của mỗi landmark (0-1)
2. Visibility: Mức độ nhìn thấy của landmark (0-1)
3. Spine Alignment Score: Độ thẳng của cột sống
4. Scapula Symmetry: Độ đối xứng giữa hai xương bả vai
5. Posture Score: Điểm tư thế tổng thể
"""


In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import math
from google.colab.patches import cv2_imshow

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
# Khởi tạo MediaPipe Pose
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

# Định nghĩa các landmarks
SHOULDER_LEFT = 11
SHOULDER_RIGHT = 12
HIP_LEFT = 23
HIP_RIGHT = 24

class SpineAnalyzer:
    """
    Class phân tích cột sống và xương bả vai
    """

    def __init__(self):
        self.spine_segments = {
            'C7': 0.0,      # Đốt sống cổ 7
            'T3': 0.167,    # Thoracic 3 (1/6)
            'T6': 0.333,    # Thoracic 6 (1/3)
            'T9': 0.5,      # Thoracic 9 (1/2)
            'T12': 0.667,   # Thoracic 12 (2/3)
            'L3': 0.833,    # Lumbar 3 (5/6)
            'S1': 1.0       # Sacrum (đáy)
        }

    def calculate_spine_points(self, landmarks, img_width, img_height):
        """
        Tính toán tọa độ các đốt cột sống

        Returns:
            dict: Tọa độ và thông tin của từng đốt sống
        """
        spine_data = {}

        # Lấy tọa độ các điểm cơ bản
        l_shoulder = landmarks[SHOULDER_LEFT]
        r_shoulder = landmarks[SHOULDER_RIGHT]
        l_hip = landmarks[HIP_LEFT]
        r_hip = landmarks[HIP_RIGHT]

        # Tính điểm giữa vai và hông
        mid_shoulder = np.array([
            (l_shoulder.x + r_shoulder.x) / 2,
            (l_shoulder.y + r_shoulder.y) / 2,
            (l_shoulder.z + r_shoulder.z) / 2
        ])

        mid_hip = np.array([
            (l_hip.x + r_hip.x) / 2,
            (l_hip.y + r_hip.y) / 2,
            (l_hip.z + r_hip.z) / 2
        ])

        # Vector cột sống
        spine_vector = mid_hip - mid_shoulder

        # Tính từng đốt sống dựa trên tỷ lệ
        for segment_name, ratio in self.spine_segments.items():
            point = mid_shoulder + spine_vector * ratio

            # Tính confidence dựa trên visibility của các landmarks xung quanh
            confidence = (l_shoulder.visibility + r_shoulder.visibility +
                         l_hip.visibility + r_hip.visibility) / 4

            spine_data[segment_name] = {
                'position': point,
                'pixel_coords': (int(point[0] * img_width), int(point[1] * img_height)),
                'confidence': confidence,
                'depth': point[2]  # z-coordinate cho độ sâu
            }

        return spine_data

    def calculate_scapula_points(self, landmarks, img_width, img_height):
        """
        Tính toán các điểm xương bả vai (scapula)

        Xương bả vai có các điểm chính:
        - Acromion: Đỉnh vai
        - Superior angle: Góc trên
        - Inferior angle: Góc dưới
        - Medial border: Mép trong (gần cột sống)
        - Lateral border: Mép ngoài
        """
        scapula_data = {}

        l_shoulder = landmarks[SHOULDER_LEFT]
        r_shoulder = landmarks[SHOULDER_RIGHT]
        l_hip = landmarks[HIP_LEFT]
        r_hip = landmarks[HIP_RIGHT]

        l_ear = landmarks[LEFT_EAR]
        r_ear = landmarks[RIGHT_EAR]
        # Tính width của vai (khoảng cách hai vai)
        shoulder_width = math.sqrt(
            (r_shoulder.x - l_shoulder.x)**2 +
            (r_shoulder.y - l_shoulder.y)**2
        )

        scapula_data['right_ear'] = {
            'position': np.array([r_ear.x, r_ear.y, r_ear.z]),
            'pixel_coords': (int(r_ear.x * img_width), int(r_ear.y * img_height)),
            'confidence': r_ear.visibility,
            'description': 'hong phai'
        }

        scapula_data['left_ear'] = {
            'position': np.array([l_ear.x, l_ear.y, l_ear.z]),
            'pixel_coords': (int(l_ear.x * img_width), int(l_ear.y * img_height)),
            'confidence': l_ear.visibility,
            'description': 'Hong phai'
        }

        scapula_data['right_hip'] = {
            'position': np.array([r_hip.x, r_hip.y, r_hip.z]),
            'pixel_coords': (int(r_hip.x * img_width), int(r_hip.y * img_height)),
            'confidence': r_hip.visibility,
            'description': 'hong phai'
        }

        scapula_data['left_hip'] = {
            'position': np.array([l_hip.x, l_hip.y, l_hip.z]),
            'pixel_coords': (int(l_hip.x * img_width), int(l_hip.y * img_height)),
            'confidence': l_hip.visibility,
            'description': 'Hong phai'
        }


        # === XƯƠNG BẢ VAI TRÁI ===

        # Acromion (đỉnh vai)
        scapula_data['left_acromion'] = {
            'position': np.array([l_shoulder.x, l_shoulder.y, l_shoulder.z]),
            'pixel_coords': (int(l_shoulder.x * img_width), int(l_shoulder.y * img_height)),
            'confidence': l_shoulder.visibility,
            'description': 'Dinh vai trai (Acromion)'
        }

        # Superior angle (góc trên) - cao hơn vai một chút, gần cột sống hơn
        sup_angle_left = np.array([
            l_shoulder.x + shoulder_width * 0.15,  # Dịch vào trong
            l_shoulder.y - 0.02,  # Dịch lên trên một chút
            l_shoulder.z
        ])
        scapula_data['left_superior_angle'] = {
            'position': sup_angle_left,
            'pixel_coords': (int(sup_angle_left[0] * img_width), int(sup_angle_left[1] * img_height)),
            'confidence': l_shoulder.visibility * 0.8,
            'description': 'Goc tren xuong ba vai trai'
        }

        # Inferior angle (góc dưới) - điểm thấp nhất của xương bả vai
        # Nằm ở ~40% đường từ vai xuống hông, lệch ra ngoài
        inf_angle_left = np.array([
            l_shoulder.x + (l_hip.x - l_shoulder.x) * 0.25,  # Lệch ra ngoài
            l_shoulder.y + (l_hip.y - l_shoulder.y) * 0.4,   # 40% xuống dưới
            l_shoulder.z + (l_hip.z - l_shoulder.z) * 0.4
        ])
        scapula_data['left_inferior_angle'] = {
            'position': inf_angle_left,
            'pixel_coords': (int(inf_angle_left[0] * img_width), int(inf_angle_left[1] * img_height)),
            'confidence': (l_shoulder.visibility + l_hip.visibility) / 2 * 0.7,
            'description': 'Goc duoi xuong ba vai trai'
        }

        # Medial border (mép trong) - điểm giữa của mép gần cột sống
        medial_left = np.array([
            l_shoulder.x + shoulder_width * 0.2,  # Gần cột sống hơn
            (sup_angle_left[1] + inf_angle_left[1]) / 2,
            l_shoulder.z
        ])
        scapula_data['left_medial_border'] = {
            'position': medial_left,
            'pixel_coords': (int(medial_left[0] * img_width), int(medial_left[1] * img_height)),
            'confidence': l_shoulder.visibility * 0.6,
            'description': 'Mep trong xuong ba vai trai'
        }

        # === XƯƠNG BẢ VAI PHẢI (đối xứng) ===

        scapula_data['right_acromion'] = {
            'position': np.array([r_shoulder.x, r_shoulder.y, r_shoulder.z]),
            'pixel_coords': (int(r_shoulder.x * img_width), int(r_shoulder.y * img_height)),
            'confidence': r_shoulder.visibility,
            'description': 'Dinh vai phai (Acromion)'
        }

        sup_angle_right = np.array([
            r_shoulder.x - shoulder_width * 0.15,
            r_shoulder.y - 0.02,
            r_shoulder.z
        ])
        scapula_data['right_superior_angle'] = {
            'position': sup_angle_right,
            'pixel_coords': (int(sup_angle_right[0] * img_width), int(sup_angle_right[1] * img_height)),
            'confidence': r_shoulder.visibility * 0.8,
            'description': 'Goc tren xuong ba vai phai'
        }

        inf_angle_right = np.array([
            r_shoulder.x + (r_hip.x - r_shoulder.x) * 0.25,
            r_shoulder.y + (r_hip.y - r_shoulder.y) * 0.4,
            r_shoulder.z + (r_hip.z - r_shoulder.z) * 0.4
        ])
        scapula_data['right_inferior_angle'] = {
            'position': inf_angle_right,
            'pixel_coords': (int(inf_angle_right[0] * img_width), int(inf_angle_right[1] * img_height)),
            'confidence': (r_shoulder.visibility + r_hip.visibility) / 2 * 0.7,
            'description': 'Goc duoi xuong ba vai phai'
        }

        medial_right = np.array([
            r_shoulder.x - shoulder_width * 0.2,
            (sup_angle_right[1] + inf_angle_right[1]) / 2,
            r_shoulder.z
        ])
        scapula_data['right_medial_border'] = {
            'position': medial_right,
            'pixel_coords': (int(medial_right[0] * img_width), int(medial_right[1] * img_height)),
            'confidence': r_shoulder.visibility * 0.6,
            'description': 'Mep trong xuong ba vai phai'
        }

        return scapula_data

    def calculate_alignment_score(self, spine_data):
        """
        Tính điểm thẳng hàng của cột sống (0-100)

        Phương pháp: Tính độ lệch chuẩn của các điểm x-coordinate
        Cột sống thẳng -> độ lệch thấp -> điểm cao
        """
        x_coords = [data['position'][0] for data in spine_data.values()]
        std_dev = np.std(x_coords)

        # Chuẩn hóa: std_dev càng nhỏ càng tốt
        # std_dev < 0.01 -> điểm 100
        # std_dev > 0.05 -> điểm giảm dần
        alignment_score = max(0, 100 - (std_dev * 2000))

        return alignment_score, std_dev

    def calculate_scapula_symmetry(self, scapula_data):
        """
        Tính độ đối xứng giữa hai xương bả vai (0-100)

        So sánh khoảng cách tương đối giữa các điểm tương ứng
        """
        # Lấy các cặp điểm tương ứng
        left_inf = scapula_data['left_inferior_angle']['position']
        right_inf = scapula_data['right_inferior_angle']['position']

        left_acr = scapula_data['left_acromion']['position']
        right_acr = scapula_data['right_acromion']['position']

        # Tính khoảng cách từ mỗi inferior angle đến acromion
        left_length = np.linalg.norm(left_inf - left_acr)
        right_length = np.linalg.norm(right_inf - right_acr)

        # Tính độ chênh lệch
        diff = abs(left_length - right_length)

        # Chuẩn hóa thành điểm (diff càng nhỏ càng tốt)
        symmetry_score = max(0, 100 - (diff * 500))

        return symmetry_score, diff

    def calculate_posture_score(self, spine_data, scapula_data, landmarks):
        """
        Tính điểm tư thế tổng thể (0-100)

        Kết hợp:
        - Độ thẳng cột sống (40%)
        - Độ đối xứng xương bả vai (30%)
        - Độ tin cậy trung bình của landmarks (30%)
        """
        alignment_score, _ = self.calculate_alignment_score(spine_data)
        symmetry_score, _ = self.calculate_scapula_symmetry(scapula_data)

        # Tính confidence trung bình
        l_shoulder = landmarks[SHOULDER_LEFT]
        r_shoulder = landmarks[SHOULDER_RIGHT]
        l_hip = landmarks[HIP_LEFT]
        r_hip = landmarks[HIP_RIGHT]

        avg_confidence = (l_shoulder.visibility + r_shoulder.visibility +
                         l_hip.visibility + r_hip.visibility) / 4
        confidence_score = avg_confidence * 100

        # Tổng hợp điểm
        posture_score = (alignment_score * 0.4 +
                        symmetry_score * 0.3 +
                        confidence_score * 0.3)

        return posture_score, {
            'alignment': alignment_score,
            'symmetry': symmetry_score,
            'confidence': confidence_score
        }

def draw_analysis(image, spine_data, scapula_data, metrics):
    """
    Vẽ phân tích lên hình ảnh
    """
    h, w = image.shape[:2]

    # Màu sắc
    SPINE_COLOR = (0, 255, 0)      # Xanh lá - cột sống
    SCAPULA_COLOR = (255, 0, 255)  # Tím - xương bả vai
    TEXT_COLOR = (255, 255, 255)   # Trắng - text

    # # === VẼ CỘT SỐNG ===
    # spine_points_list = list(spine_data.keys())
    # for i in range(len(spine_points_list) - 1):
    #     pt1 = spine_data[spine_points_list[i]]['pixel_coords']
    #     pt2 = spine_data[spine_points_list[i + 1]]['pixel_coords']
    #     cv2.line(image, pt1, pt2, SPINE_COLOR, 3)

    # # Vẽ các điểm cột sống
    # for name, data in spine_data.items():
    #     pt = data['pixel_coords']
    #     cv2.circle(image, pt, 6, SPINE_COLOR, -1)
    #     cv2.putText(image, name, (pt[0] + 10, pt[1] - 5),
    #                cv2.FONT_HERSHEY_SIMPLEX, 0.5, SPINE_COLOR, 2)

    # === VẼ XƯƠNG BẢ VAI ===
    # # Xương bả vai trái
    # cv2.line(image,
    #         scapula_data['left_superior_angle']['pixel_coords'],
    #         scapula_data['left_inferior_angle']['pixel_coords'],
    #         SCAPULA_COLOR, 2)
    # cv2.line(image,
    #         scapula_data['left_acromion']['pixel_coords'],
    #         scapula_data['left_inferior_angle']['pixel_coords'],
    #         SCAPULA_COLOR, 2)
    # cv2.line(image,
    #         scapula_data['left_superior_angle']['pixel_coords'],
    #         scapula_data['left_medial_border']['pixel_coords'],
    #         SCAPULA_COLOR, 2)
    # cv2.line(image,
    #         scapula_data['left_medial_border']['pixel_coords'],
    #         scapula_data['left_inferior_angle']['pixel_coords'],
    #         SCAPULA_COLOR, 2)

    # # Xương bả vai phải
    # cv2.line(image,
    #         scapula_data['right_superior_angle']['pixel_coords'],
    #         scapula_data['right_inferior_angle']['pixel_coords'],
    #         SCAPULA_COLOR, 2)
    # cv2.line(image,
    #         scapula_data['right_acromion']['pixel_coords'],
    #         scapula_data['right_inferior_angle']['pixel_coords'],
    #         SCAPULA_COLOR, 2)
    # cv2.line(image,
    #         scapula_data['right_superior_angle']['pixel_coords'],
    #         scapula_data['right_medial_border']['pixel_coords'],
    #         SCAPULA_COLOR, 2)
    # cv2.line(image,
    #         scapula_data['right_medial_border']['pixel_coords'],
    #         scapula_data['right_inferior_angle']['pixel_coords'],
    #         SCAPULA_COLOR, 2)

    # for name, data in scapula_data.items():
    #     pt = data['pixel_coords']
    #     cv2.circle(image, pt, 5, SCAPULA_COLOR, -1)

    cv2.circle(image,scapula_data['right_acromion']['pixel_coords'], 5, SCAPULA_COLOR, -1)
    cv2.circle(image,scapula_data['left_acromion']['pixel_coords'], 5, SCAPULA_COLOR, -1)
    cv2.circle(image,scapula_data['right_hip']['pixel_coords'], 5, SCAPULA_COLOR, -1)
    cv2.circle(image,scapula_data['left_hip']['pixel_coords'], 5, SCAPULA_COLOR, -1)
    cv2.circle(image,scapula_data['left_ear']['pixel_coords'], 5, SCAPULA_COLOR, -1)
    cv2.circle(image,scapula_data['right_ear']['pixel_coords'], 5, SCAPULA_COLOR, -1)


    PoseLandmark = mp_pose.PoseLandmark
    for lm in PoseLandmark:
        print(lm.name, lm.value)


    # # === HIỂN THỊ THÔNG SỐ ===
    # y_offset = 30
    # cv2.putText(image, "=== THONG SO DANH GIA ===",
    #            (10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.6, TEXT_COLOR, 2)
    # y_offset += 30

    # cv2.putText(image, f"Posture Score: {metrics['posture_score']:.1f}/100",
    #            (10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.5, TEXT_COLOR, 1)
    # y_offset += 25

    # cv2.putText(image, f"Spine Alignment: {metrics['components']['alignment']:.1f}/100",
    #            (10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.5, TEXT_COLOR, 1)
    # y_offset += 25

    # cv2.putText(image, f"Scapula Symmetry: {metrics['components']['symmetry']:.1f}/100",
    #            (10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.5, TEXT_COLOR, 1)
    # y_offset += 25

    # cv2.putText(image, f"Detection Confidence: {metrics['components']['confidence']:.1f}/100",
    #            (10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.5, TEXT_COLOR, 1)
    # y_offset += 25

    # cv2.putText(image, f"Spine Deviation: {metrics['spine_deviation']:.4f}",
    #            (10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.5, TEXT_COLOR, 1)
    # y_offset += 25

    # cv2.putText(image, f"Scapula Diff: {metrics['scapula_diff']:.4f}",
    #            (10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.5, TEXT_COLOR, 1)

    return image

def process_image(image_path):
    """
    Xử lý một ảnh RGB
    """
    # Đọc ảnh
    image = cv2.imread(image_path)

    if image is None:
        print("Không thể đọc ảnh!")
        return

    # Chuyển BGR sang RGB cho MediaPipe
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    # Khởi tạo analyzer
    analyzer = SpineAnalyzer()

    # Xử lý với MediaPipe Pose
    with mp_pose.Pose(
        static_image_mode=True,
        model_complexity=2,
        min_detection_confidence=0.5) as pose:

        results = pose.process(image_rgb)

        if not results.pose_landmarks:
            print("Không phát hiện được pose trong ảnh!")
            return

        # Vẽ skeleton cơ bản
        # mp_drawing.draw_landmarks(
        #     image,
        #     results.pose_landmarks,
        #     mp_pose.POSE_CONNECTIONS,
        #     landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style())

        # Tính toán các điểm cột sống
        spine_data = analyzer.calculate_spine_points(
            results.pose_landmarks.landmark,
            image.shape[1],
            image.shape[0])

        # Tính toán các điểm xương bả vai
        scapula_data = analyzer.calculate_scapula_points(
            results.pose_landmarks.landmark,
            image.shape[1],
            image.shape[0])

        # Tính các thông số đánh giá
        alignment_score, spine_deviation = analyzer.calculate_alignment_score(spine_data)
        symmetry_score, scapula_diff = analyzer.calculate_scapula_symmetry(scapula_data)
        posture_score, components = analyzer.calculate_posture_score(
            spine_data, scapula_data, results.pose_landmarks.landmark)

        metrics = {
            'posture_score': posture_score,
            'components': components,
            'spine_deviation': spine_deviation,
            'scapula_diff': scapula_diff
        }

        # Vẽ phân tích lên ảnh
        result_image = draw_analysis(image, spine_data, scapula_data, metrics)

        # Hiển thị
        cv2_imshow(result_image)
        cv2_imshow(result_image)

        # # In chi tiết ra console
        # print("\n" + "="*60)
        # print("PHÂN TÍCH CHI TIẾT")
        # print("="*60)

        # print("\n--- CỘT SỐNG ---")
        # for name, data in spine_data.items():
        #     print(f"{name}: Confidence={data['confidence']:.3f}, Depth={data['depth']:.3f}")

        # print("\n--- XƯƠNG BẢ VAI ---")
        # for name, data in scapula_data.items():
        #     print(f"{name}: {data['description']}")
        #     print(f"  Confidence: {data['confidence']:.3f}")

        # print("\n--- THÔNG SỐ ĐÁNH GIÁ ---")
        # print(f"Posture Score: {posture_score:.2f}/100")
        # print(f"  - Spine Alignment: {components['alignment']:.2f}/100")
        # print(f"  - Scapula Symmetry: {components['symmetry']:.2f}/100")
        # print(f"  - Detection Confidence: {components['confidence']:.2f}/100")
        # print(f"Spine Deviation: {spine_deviation:.5f}")
        # print(f"Scapula Difference: {scapula_diff:.5f}")

        # print("\n--- ĐÁNH GIÁ ---")
        # if posture_score >= 80:
        #     print("✓ Tư thế RẤT TỐT!")
        # elif posture_score >= 60:
        #     print("○ Tư thế KHÁ TỐT, có thể cải thiện")
        # else:
        #     print("✗ Tư thế CẦN ĐIỀU CHỈNH")

        cv2.waitKey(0)
        cv2.destroyAllWindows()



In [ ]:
# CHẠY CHƯƠNG TRÌNH
if __name__ == "__main__":
    print("""
    ╔═══════════════════════════════════════════════════════════╗
    ║  MEDIAPIPE POSE - PHÂN TÍCH CỘT SỐNG VÀ XƯƠNG BẢ VAI    ║
    ╚═══════════════════════════════════════════════════════════╝
    """)

    image_path = "/content/drive/MyDrive/scoliosis.v1i.voc/train/IMG_1295_PNG.rf.20baa820a457193d987cfb1566741bdb.jpg"
    process_image(image_path)